In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print(
        "There might be a problem with your API key? Please visit the troubleshooting notebook!"
    )

MODEL = "gpt-5-nano"
openai = OpenAI()


API key looks good so far


In [4]:
links = fetch_website_links("https://www.budgetbytes.com/category/recipes/one-pot/")
links


['/join/',
 '#main-content',
 'https://www.budgetbytes.com',
 '/join/',
 '/join/',
 'https://www.budgetbytes.com/?memberful_endpoint=auth',
 'https://www.budgetbytes.com/category/recipes/',
 'https://www.budgetbytes.com/category/recipes/#recent',
 'https://www.budgetbytes.com/index/',
 'https://www.budgetbytes.com/index/#recipes-by-ingredient',
 'https://www.budgetbytes.com/random/',
 'https://www.budgetbytes.com/category/extra-bytes/recipe-roundups/',
 '/category/recipes/',
 'https://www.budgetbytes.com/category/extra-bytes/budget-friendly-meal-prep/',
 'https://www.budgetbytes.com/19-quick-and-easy-weeknight-dinners/',
 'https://www.budgetbytes.com/category/recipes/one-pot/',
 'https://www.budgetbytes.com/category/recipes/meat/chicken/',
 'https://www.budgetbytes.com/category/recipes/pasta/',
 'https://www.budgetbytes.com/category/recipes/vegetarian/',
 'https://www.budgetbytes.com/category/recipes/salads/',
 'https://www.budgetbytes.com/category/recipes/breakfast/',
 'https://www.bu

In [5]:
link_system_prompt = """
You are given a list of links from a cooking website. Pick the 3 recipes easiest for a COMPLETE BEGINNER, ranked by (in order):
1. Short total time (ideally <30 min)
2. Only basic equipment (pan, pot, oven, knife, stovetop — no stand mixer, food processor, thermometer, pressure cooker, sous vide, etc.)
3. Few, common ingredients
4. Simple techniques (no deboning, folding, tempering, proofing, laminating, multi-stage cooking)

Ignore non-recipe links (category, blog, about, shop, login, legal, email).

Respond in JSON:
{"recipes":[
  {"rank":1,"title":"One-Pan Garlic Butter Pasta","url":"https://...","reason":"15 min, one pan, 5 ingredients"},
  {"rank":2,"title":"Scrambled Eggs on Toast","url":"https://...","reason":"10 min, basic pan, no knife skills"},
  {"rank":3,"title":"Microwave Mug Oatmeal","url":"https://...","reason":"3 min, microwave, 3 ingredients"}
]}
"""


In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the cooking website {url} -
Please decide the 3 EASIEST recipes for a complete beginner to make, prioritising fast cook time and minimal kitchen equipment. Respond with the full https URL in JSON format.
Do not include category pages, blog posts that aren't recipes, login/register pages, Terms of Service, Privacy, or email links.
Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [8]:
print(get_links_user_prompt("https://www.budgetbytes.com/category/recipes/one-pot/"))



Here is the list of links on the cooking website https://www.budgetbytes.com/category/recipes/one-pot/ -
Please decide the 3 EASIEST recipes for a complete beginner to make, prioritising fast cook time and minimal kitchen equipment. Respond with the full https URL in JSON format.
Do not include category pages, blog posts that aren't recipes, login/register pages, Terms of Service, Privacy, or email links.
Links (some might be relative links):

/join/
#main-content
https://www.budgetbytes.com
/join/
/join/
https://www.budgetbytes.com/?memberful_endpoint=auth
https://www.budgetbytes.com/category/recipes/
https://www.budgetbytes.com/category/recipes/#recent
https://www.budgetbytes.com/index/
https://www.budgetbytes.com/index/#recipes-by-ingredient
https://www.budgetbytes.com/random/
https://www.budgetbytes.com/category/extra-bytes/recipe-roundups/
/category/recipes/
https://www.budgetbytes.com/category/extra-bytes/budget-friendly-meal-prep/
https://www.budgetbytes.com/19-quick-and-easy-w

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['recipes'])} relevant links")
    return links


In [13]:
select_relevant_links("https://www.budgetbytes.com/category/recipes/one-pot/")


Selecting relevant links for https://www.budgetbytes.com/category/recipes/one-pot/ by calling gpt-5-nano
Found 3 relevant links


{'recipes': [{'rank': 1,
   'title': 'One-Pot Veggie Pasta',
   'url': 'https://www.budgetbytes.com/one-pot-veggie-pasta/',
   'reason': 'About 20-25 minutes, one pot, few common ingredients, simple sauté and simmer.'},
  {'rank': 2,
   'title': 'Skillet Cheeseburger Pasta',
   'url': 'https://www.budgetbytes.com/skillet-cheeseburger-pasta/',
   'reason': 'About 25-30 minutes, one pan (skillet), minimal ingredients, straightforward browning and mixing.'},
  {'rank': 3,
   'title': 'Beef Taco Pasta',
   'url': 'https://www.budgetbytes.com/beef-taco-pasta/',
   'reason': 'About 25-30 minutes, one pot, simple taco-seasoned beef with pasta and sauce.'}]}

## Making the brochure with the cooking recipes.


In [14]:
# format the user prompt first.
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Top 3 Recipes:\n"
    for link in relevant_links["recipes"]:
        result += f"\n\n### Recipe name: {link['title']}\n"
        result += f"\n\n### Reason for selection: {link['reason']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [15]:
print(
    fetch_page_and_all_relevant_links(
        "https://www.budgetbytes.com/category/recipes/one-pot/"
    )
)


Selecting relevant links for https://www.budgetbytes.com/category/recipes/one-pot/ by calling gpt-5-nano
Found 3 relevant links
## Landing Page:

50+ One Pot Meals - Easy Dinner Ideas - Budget Bytes

Sign up
for access to our weekly meal plans
Skip to content
Sign up
for access to our weekly meal plans
Sign Up
Login
Recipes
Recipe Search
Categories
Ingredient Index
Surprise Me
Recipe Roundups
Popular
Meal Prep
Dinner Ideas
One Pot Meals
Chicken Recipes
Pasta Recipes
Vegetarian Recipes
Salad Recipes
Breakfast Recipes
Recipes under $10
Sandwich Recipes
Soup Recipes
Air Fryer Recipes
Meal Plans
About
About Budget Bytes
New? Start Here
FAQ
Contact
Search for
Recipes
»
One Pot Meals
One Pot Meals
These simple One Pot Meals are your answer to quick and easy weeknight dinners! Everything cooks in one pot or pan for maximum flavor and minimum cleanup! These easy dinner recipes eliminate the need to figure out what sides to cook with your main dish, because each recipes provides your protein, v

In [19]:
cookbook_system_prompt = """
You produce a short, friendly "Beginner's Cookbook" in markdown from 3 pre-selected easy recipes (raw scraped content supplied).

For EACH recipe, output a section with:
- A one-line confidence-boosting intro
- **Total time** and **servings**
- **Equipment** (basics only)
- **Ingredients** as bullets, with common substitutions in brackets
- **Steps**: numbered, plain encouraging language; define any cooking term on first use (e.g. "sauté = cook in a little oil over medium heat until soft")
- **Beginner tip**: the single most common mistake to avoid

End with "What to cook first" — pick ONE recipe as the best starting point, one-sentence reason.

Be warm, concise, never condescending. Do NOT invent ingredients or steps beyond the scraped content — only rephrase and clarify.
"""


In [16]:
def get_cookbook_user_prompt(url):
    user_prompt = f"""
Here are 3 beginner-friendly recipes scraped from {url}. Please turn them into a short Beginner's Cookbook in markdown, following the format in your system prompt.
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[
        :5_000
    ]  # Truncate if more than 5,000 characters (can use 5000, 5_000 is for readability)
    return user_prompt

In [ ]:
get_cookbook_user_prompt("https://www.budgetbytes.com/category/recipes/one-pot/")


### Stream the cookbook


In [22]:
def stream_cookbook(url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": cookbook_system_prompt},
            {"role": "user", "content": get_cookbook_user_prompt(url)},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)


In [23]:
stream_cookbook("https://www.budgetbytes.com/category/recipes/one-pot/")

Selecting relevant links for https://www.budgetbytes.com/category/recipes/one-pot/ by calling gpt-5-nano
Found 3 relevant links


# Beginner's Cookbook: Easy One-Pot Meals

Welcome to your first step in simple, delicious cooking! These recipes are designed to build your confidence with easy ingredients and straightforward steps — all in one pot for minimal mess. Let’s get cooking!

---

## One-Pot Veggie Pasta

This veggie-packed pasta is a colorful, comforting meal that’s easy to customize with what you have in your fridge.

- **Total time:** 30 minutes  
- **Servings:** 4  

### Equipment
- Large pot or deep skillet (with lid if possible)  
- Wooden spoon or spatula  

### Ingredients
- Pasta (any shape like penne or fusilli)  
- 2-3 cups mixed vegetables (carrots, bell peppers, mushrooms, broccoli, sundried tomatoes — add whatever you have!)  
- Vegetable broth or water (can substitute with chicken broth)  
- Olive oil  
- Salt and pepper to taste  
- Optional: chickpeas or squash pieces for extra protein and texture  

### Steps
1. Heat a bit of olive oil in your pot over medium heat.  
2. Add chopped vegetables and sauté (cook in a little oil over medium heat until soft) for about 5 minutes.  
3. Add pasta and enough broth or water to just cover it.  
4. Bring to a boil, then reduce heat to a simmer (small bubbles gently rising) and cover. Cook until pasta is tender, about 10-15 minutes. Check occasionally.  
5. Stir occasionally to prevent sticking and add more liquid if needed.  
6. Season with salt and pepper, and stir in any extras like chickpeas.  
7. Serve warm and enjoy!  

### Beginner tip
Avoid overfilling the pot with liquid — add just enough to cover the pasta so it cooks perfectly without becoming mushy.

---

## One-Pot Creamy Sun-Dried Tomato Pasta

This creamy pasta is a pantry-friendly way to enjoy rich flavors with just one pot and simple ingredients.

- **Total time:** 30 minutes  
- **Servings:** 4  

### Equipment
- Large pot or deep skillet  
- Wooden spoon or spatula  

### Ingredients
- Pasta (any kind you prefer)  
- Sun-dried tomatoes (use jarred in oil or dry-packed, soaked if dry)  
- Garlic (fresh or powdered)  
- Heavy cream or half and half (milk can be used as a lighter substitute)  
- Olive oil or butter  
- Parmesan or another hard cheese (optional)  
- Salt and pepper  

### Steps
1. Warm olive oil or butter in the pot over medium heat.  
2. Add garlic and sun-dried tomatoes, sauté until fragrant (about 1-2 minutes).  
3. Pour in pasta and enough water or broth to cover. Bring to a boil, then lower to a simmer and cover.  
4. Cook until pasta is tender, stirring occasionally and adding liquid as needed (about 10-15 minutes).  
5. Lower heat and stir in cream to create the sauce. Warm through without boiling.  
6. Add grated cheese if using, season with salt and pepper, and stir until creamy and delicious.  
7. Serve immediately.  

### Beginner tip  
Don’t add cream until the pasta is cooked—adding it too early can prevent the pasta from cooking properly.

---

## One-Pot Chicken and Rice

A classic, hearty dish that’s simple to prepare and comes together in just one pot.

- **Total time:** about 40 minutes  
- **Servings:** 4  

### Equipment
- Large pot or deep skillet with a lid  
- Wooden spoon or spatula  

### Ingredients
- Chicken (breasts or thighs, cut into pieces)  
- Rice (white long-grain or basmati)  
- Onion and garlic  
- Chicken broth  
- Olive oil  
- Salt, pepper, and your favorite herbs (like thyme or parsley)  

### Steps
1. Heat olive oil in the pot and brown chicken pieces on all sides until golden. Remove and set aside.  
2. In the same pot, sauté chopped onion and garlic until softened (about 3-4 minutes).  
3. Add rice and stir to coat with the oil and aromatics.  
4. Pour in chicken broth and return chicken to the pot. Season with salt, pepper, and herbs.  
5. Bring to a boil, then lower heat to a simmer and cover. Cook until rice is tender and chicken is cooked through (about 20 minutes).  
6. Fluff rice and serve warm.  

### Beginner tip  
Resist the urge to stir rice while it cooks; stirring breaks the grains and makes it mushy.

---

## What to cook first?

Start with the **One-Pot Veggie Pasta** — it’s forgiving, flexible, and a perfect way to build confidence with one-pot cooking using simple ingredients you likely already have. Enjoy your kitchen adventure!